# Buổi H1: Lưu và nạp mô hình, mang mô hình ra khỏi notebook

**Hiện trường.** Chị Hạnh đã có mô hình đoán khách có quay lại cửa hàng không. Nó chạy tốt trong notebook. Nhưng nhân viên cần dùng nó **mỗi sáng**, trên máy khác, không mở notebook, không học lại từ đầu. Chị hỏi: *làm sao "đóng gói" mô hình để mang đi?*

Buổi này dạy **lưu mô hình ra file** và **nạp lại** bằng `joblib`, cùng những thứ đi kèm để việc nạp lại **đáng tin**: ghi chú phiên bản, bài kiểm tra "ví dụ vàng", và cảnh báo về an toàn khi nạp file lạ.

**Ẩn dụ:** nấu một món ngon rồi muốn người khác làm lại. Ta không đưa nồi canh nóng, mà đưa **công thức đầy đủ** (nguyên liệu, cách sơ chế, nhiệt độ) và một **món mẫu để so vị**. Mô hình đóng gói cũng vậy: gồm cả bước chế biến dữ liệu (pipeline), ghi chú phiên bản, và vài dòng mẫu để kiểm tra kết quả sau khi nạp.

**Mindset của người làm dữ liệu:** mô hình không phải là cục trọng số; mô hình là **toàn bộ chuỗi từ bảng thô đến dự đoán**. Lưu thiếu một mắt xích là lỗi âm thầm.

**Lưu ý:** dữ liệu khách hàng dựng có chủ đích (không có bộ dữ liệu khách hàng thật trong sklearn); dùng để học.

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 0 | Thấy ngay: học, lưu, nạp, dự đoán giống hệt |
| Chặng 1 | Trong file có gì, và ghi chú đi kèm |
| Chặng 2 | Ví dụ vàng: bài kiểm tra sau khi nạp |
| Chặng 3 | Bẫy: lưu mô hình mà quên pipeline |
| Chặng 4 | An toàn: đừng nạp file lạ |
| Chặng 5 | Hàm nạp mô hình có kiểm tra |

## Chặng 0: Thấy ngay, học, lưu, nạp

**Câu hỏi:** mô hình nạp lại từ file có cho dự đoán giống hệt mô hình gốc không?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
def make_customers(n, seed, month=0):
    """Synthetic customer table (built on purpose for teaching). month > 0 makes later data drift."""
    r = np.random.RandomState(seed)
    tenure = r.gamma(3.0, 6.0, n).round(0).clip(1, 72)
    plan = r.choice(["basic", "plus", "premium"], n, p=[0.55 - 0.03 * month, 0.30, 0.15 + 0.03 * month])
    city = r.choice(["HCM", "HN", "DN", "CT"], n, p=[0.4, 0.3, 0.15, 0.15])
    spend = (np.where(plan == "basic", 0.8, np.where(plan == "plus", 1.6, 3.0)) * r.lognormal(0, 0.35, n) * (1 + 0.04 * month)).round(2)
    visits = r.poisson(np.clip(2 + 0.12 * tenure ** 0.7 + 0.5 * spend, 0.5, None)).astype(int)
    complaints = r.poisson(0.35 + 0.002 * month, n).astype(int)
    z = -0.2 + 0.035 * tenure + 0.32 * visits - 0.55 * complaints + 0.25 * spend - 1.6
    if month:
        z = z - 0.12 * month * (plan == "premium")
    returned = (r.rand(n) < 1 / (1 + np.exp(-z))).astype(int)
    df = pd.DataFrame({"tenure_months": tenure, "monthly_spend": spend, "visits_last_30d": visits, "complaints": complaints, "plan": plan, "city": city, "returned": returned})
    miss = r.rand(n) < 0.04
    df.loc[miss, "monthly_spend"] = np.nan
    return df

import tempfile, joblib, os
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
df = make_customers(3000, seed=0)
X, y = df.drop(columns="returned"), df["returned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
num_cols = ["tenure_months", "monthly_spend", "visits_last_30d", "complaints"]
cat_cols = ["plan", "city"]
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
workdir = tempfile.mkdtemp()
path = os.path.join(workdir, "customer_model.joblib")
joblib.dump(model, path)
loaded = joblib.load(path)
same = bool(np.array_equal(model.predict_proba(X_test), loaded.predict_proba(X_test)))
print("test accuracy:", round(model.score(X_test, y_test), 3), "| file size (KB):", round(os.path.getsize(path) / 1024, 1))
print("loaded model gives identical probabilities:", same)

test accuracy: 0.665 | file size (KB): 4.0
loaded model gives identical probabilities: True


**Giải thích từng dòng**

- `def make_customers(n, seed, month=0)`: hàm dựng bảng khách hàng giả: thời gian gắn bó, chi tiêu, số lần ghé, số khiếu nại, gói, thành phố, và nhãn `returned` (1 là quay lại). Có vài ô trống ở cột chi tiêu như dữ liệu thật.
- `ColumnTransformer([...])`: chế biến theo cột (B4): cột số điền ô trống bằng trung vị rồi chuẩn hóa; cột chữ chuyển thành các cột 0 hoặc 1. `handle_unknown="ignore"` để gặp giá trị chữ lạ thì không báo lỗi (chặng 3 của buổi H2).
- `Pipeline([("prep", prep), ("clf", LogisticRegression())])`: **một đối tượng duy nhất** gồm cả chế biến lẫn mô hình (D3). Đây là thứ ta sẽ lưu.
- `tempfile.mkdtemp()`: tạo một thư mục tạm để thử lưu file mà không làm bừa máy.
- `joblib.dump(model, path)`: ghi cả pipeline đã học ra một file.
- `joblib.load(path)`: đọc file ra thành một pipeline y hệt, sẵn sàng dự đoán.

**Cách đọc.**
- **Con số:** độ chính xác trên tập thi **66,5%**; file nặng **4,0** KB. Xác suất của mô hình nạp lại **giống hệt** mô hình gốc.
- **Bức tranh:** chỉ cần một dòng để lưu, một dòng để nạp. Điều quan trọng là ta lưu **cả pipeline**, không chỉ bộ trọng số: người dùng đưa bảng thô (có ô trống, có cột chữ) và nhận dự đoán.
- **Câu hỏi tiếp:** trong file có gì, và làm sao biết file này tạo từ phiên bản nào?

**Mang về chặng 1:** lưu cả pipeline, nạp lại cho kết quả giống hệt.

## Chặng 1: Trong file có gì, và ghi chú đi kèm

**Câu hỏi:** nếu máy khác cài phiên bản thư viện khác, mô hình nạp lên có còn đúng?

In [3]:
import sklearn, json, datetime
meta = {
    "name": "customer_return_v1",
    "trained_at": datetime.date.today().isoformat(),
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "input_columns": list(X.columns),
    "target": "returned (1 = customer comes back)",
    "train_rows": int(len(X_train)),
    "test_accuracy": round(float(model.score(X_test, y_test)), 4),
}
meta_path = os.path.join(workdir, "customer_model.meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))
print("pipeline steps:", [name for name, _ in loaded.steps], "| classifier coefficients shape:", loaded.named_steps["clf"].coef_.shape)

{
  "name": "customer_return_v1",
  "trained_at": "2026-11-03",
  "sklearn_version": "1.9.1",
  "numpy_version": "2.5.3",
  "input_columns": [
    "tenure_months",
    "monthly_spend",
    "visits_last_30d",
    "complaints",
    "plan",
    "city"
  ],
  "target": "returned (1 = customer comes back)",
  "train_rows": 2250,
  "test_accuracy": 0.6653
}
pipeline steps: ['prep', 'clf'] | classifier coefficients shape: (1, 11)


**Giải thích từng dòng**

- `sklearn.__version__`: số phiên bản sklearn lúc huấn luyện. File `joblib` chỉ chắc chắn nạp đúng với **cùng phiên bản** (hoặc rất gần).
- `"input_columns": list(X.columns)`: danh sách cột đầu vào mà mô hình mong đợi: để kiểm tra đầu vào về sau (buổi H2).
- `json.dump(meta, f, ...)`: ghi ghi chú ra file `.meta.json` đặt cạnh file mô hình, đọc được bằng mắt và bằng code.
- `loaded.named_steps["clf"].coef_.shape`: nhìn vào bên trong: số hệ số của hồi quy logistic bằng số cột sau chế biến (4 cột số + các cột one-hot).

**Cách đọc.**
- **Con số:** ghi chú gồm phiên bản sklearn **1.9.1**, **6** cột đầu vào, độ chính xác lúc lưu **66,5%**. Bên trong pipeline có các bước **['prep', 'clf']**; hồi quy logistic có **11** hệ số.
- **Bức tranh:** file mô hình là cái hộp kín; ghi chú là nhãn dán lên hộp. Khi lỗi xảy ra sau vài tháng, câu hỏi đầu tiên luôn là "mô hình này học bằng phiên bản nào, trên bao nhiêu dữ liệu, điểm bao nhiêu". Nếu không ghi thì không ai biết.
- **Quy tắc đọc:** luôn lưu kèm: tên và ngày, phiên bản thư viện, danh sách cột, điểm thi, số dòng học. Gọn nhưng cứu được nhiều giờ truy lỗi.

**Mang về chặng 2:** lưu mô hình kèm ghi chú phiên bản và cột.

## Chặng 2: Ví dụ vàng

**Câu hỏi:** làm sao chắc rằng sau khi nạp ở máy khác, mô hình vẫn đoán đúng như ngày lưu?

In [4]:
golden = X_test.iloc[:5].copy()
golden_expected = model.predict_proba(golden)[:, 1].round(6)
golden_path = os.path.join(workdir, "golden_examples.json")
pd.concat([golden.reset_index(drop=True), pd.Series(golden_expected, name="expected_proba")], axis=1).to_json(golden_path, orient="records")
def check_golden(m, path, tol=1e-6):
    rows = pd.read_json(golden_path, orient="records")
    expected = rows.pop("expected_proba").to_numpy()
    got = m.predict_proba(rows[meta["input_columns"]])[:, 1]
    return bool(np.allclose(got, expected, atol=tol)), float(np.abs(got - expected).max())
ok, worst = check_golden(joblib.load(path), golden_path)
print("golden check after reload:", ok, "| largest difference:", worst)
broken = joblib.load(path)
broken.named_steps["clf"].coef_ = broken.named_steps["clf"].coef_ * 1.02
ok2, worst2 = check_golden(broken, golden_path)
print("golden check on a model whose weights were changed by 2%:", ok2, "| largest difference:", round(worst2, 5))

golden check after reload: True | largest difference: 4.411514060165267e-07
golden check on a model whose weights were changed by 2%: False | largest difference: 0.00395


**Giải thích từng dòng**

- `golden = X_test.iloc[:5]`: **ví dụ vàng**: 5 dòng mẫu cố định, kèm xác suất mô hình đã đoán lúc lưu.
- `np.allclose(got, expected, atol=tol)`: so xác suất đoán hôm nay với xác suất đã ghi, cho phép sai khác cực nhỏ do làm tròn.
- `coef_ * 1.02`: cố ý làm hỏng nhẹ mô hình (đổi trọng số 2%) để thấy bài kiểm tra có bắt được không.

**Cách đọc.**
- **Con số:** mô hình nạp lại qua bài kiểm tra: **True** (chênh lớn nhất 4.4e-07). Mô hình bị chỉnh trọng số 2%: **False** (chênh lớn nhất **0,00395**).
- **Bức tranh:** bài kiểm tra ví dụ vàng chạy trong vài mili giây nhưng bắt được cả thay đổi nhỏ do khác phiên bản thư viện, file hỏng, hoặc nạp nhầm mô hình. Giống bước "nếm thử món mẫu" trước khi mở bán.
- **Quy tắc đọc:** mọi lần nạp mô hình trên máy mới đều chạy ví dụ vàng trước khi dùng. Một chênh lệch nhỏ hơn dung sai thì bỏ qua; lớn hơn thì dừng lại điều tra.

**Mang về chặng 3:** ví dụ vàng là bài kiểm tra rẻ nhất sau mỗi lần nạp.

## Chặng 3: Bẫy, lưu mô hình mà quên pipeline

**Câu hỏi:** nếu chỉ lưu bộ phân loại, còn chế biến dữ liệu làm riêng, chuyện gì xảy ra ở nơi dùng?

In [5]:
# the "wrong" way: preprocess by hand in the notebook, save only the classifier
X_prep = prep.fit(X_train).transform(X_train)
clf_only = LogisticRegression(max_iter=1000).fit(X_prep, y_train)
clf_path = os.path.join(workdir, "classifier_only.joblib")
joblib.dump(clf_only, clf_path)
clf_loaded = joblib.load(clf_path)
# at the new place: someone forgets the scaler / encoder and gives numbers only
numbers_only = X_test[num_cols].fillna(X_train[num_cols].median())
try:
    clf_loaded.predict(numbers_only.to_numpy())
    result = "ran without error"
except Exception as e:
    result = type(e).__name__ + ": " + str(e)[:90]
print("classifier-only, fed the 4 raw numeric columns ->", result)
# a more subtle failure: right shape, wrong scale
fake = np.zeros((len(X_test), clf_loaded.coef_.shape[1])); fake[:, :4] = numbers_only.to_numpy()
wrong_acc = float((clf_loaded.predict(fake) == y_test).mean())
print("classifier-only, right number of columns but no scaling/encoding -> accuracy:", round(wrong_acc, 3), "| majority-class baseline:", round(float(max(y_test.mean(), 1 - y_test.mean())), 3))
print("whole pipeline from the file                                      -> accuracy:", round(loaded.score(X_test, y_test), 3))

classifier-only, fed the 4 raw numeric columns -> ValueError: X has 4 features, but LogisticRegression is expecting 11 features as input.
classifier-only, right number of columns but no scaling/encoding -> accuracy: 0.535 | majority-class baseline: 0.535
whole pipeline from the file                                      -> accuracy: 0.665


**Giải thích từng dòng**

- `prep.fit(X_train).transform(X_train)`: cách làm **sai**: chế biến bằng tay ngoài pipeline, rồi chỉ huấn luyện và lưu bộ phân loại.
- `clf_loaded.predict(numbers_only.to_numpy())`: ở nơi dùng, người ta chỉ có bộ phân loại và bảng số thô: thiếu cột one-hot nên báo lỗi số cột.
- `fake[:, :4] = numbers_only`: trường hợp nguy hiểm hơn: ai đó "chữa" lỗi bằng cách điền thêm số 0 cho đủ cột. Máy chạy được, **không báo lỗi gì**, nhưng số chưa chuẩn hóa nên dự đoán sai.

**Cách đọc.**
- **Con số:** đưa bảng số thô vào bộ phân loại riêng: **ValueError: X has 4 features, but LogisticRegression is expecting 11 features as input.**. Điền 0 cho đủ cột thì không báo lỗi, độ chính xác **53,5%** (mốc luôn đoán nhóm đông: 53,5%). Pipeline đầy đủ: **66,5%**.
- **Bức tranh:** lỗi to thì dễ thấy; lỗi **chạy được mà sai** mới nguy hiểm vì không ai biết. Pipeline gói chế biến và mô hình lại với nhau (D3) chính là để loại lỗi này.
- **Quy tắc đọc:** lưu và nạp **cả pipeline**, đưa vào bảng thô y như lúc huấn luyện.

**Mang về chặng 4:** không bao giờ tách mô hình khỏi bước chế biến dữ liệu của nó.

## Chặng 4: An toàn, đừng nạp file lạ

**Câu hỏi:** nạp một file `.joblib` có thể làm gì ngoài việc cho ra mô hình?

In [6]:
import pickle
class Harmless:
    """Shows that loading runs code. This one only builds a string; a bad file could run anything."""
    def __reduce__(self):
        return (str.upper, ("this text was produced while loading the file",))
blob = pickle.dumps(Harmless())
result = pickle.loads(blob)
print("object we got back is not a Harmless instance, it is:", type(result).__name__, "->", result)
print("sklearn models are stored with the same mechanism (pickle):", type(joblib.load(path)).__name__)

object we got back is not a Harmless instance, it is: str -> THIS TEXT WAS PRODUCED WHILE LOADING THE FILE
sklearn models are stored with the same mechanism (pickle): Pipeline


**Giải thích từng dòng**

- `def __reduce__(self)`: cách một đối tượng nói với pickle: "khi nạp tôi lại, hãy gọi hàm này với các đối số này". Hàm có thể là **bất kỳ hàm nào**.
- `(str.upper, ("...",))`: ở đây ta chọn một hàm vô hại (đổi chữ hoa). Kẻ xấu có thể chọn hàm xóa file hoặc gửi dữ liệu đi.
- `pickle.loads(blob)`: nạp lại: hàm được chạy ngay lúc nạp, trước cả khi ta gọi bất kỳ lệnh nào trên đối tượng.
- `joblib.load(path)`: `joblib` dựa trên cơ chế pickle, nên cùng một rủi ro.

**Cách đọc.**
- **Con số:** thứ nhận lại sau khi nạp là **str** với nội dung "THIS TEXT WAS PRODUCED WHILE LOADING THE FILE", chứ không phải đối tượng gốc. Tức là code đã chạy ngay lúc nạp.
- **Bức tranh:** file mô hình không chỉ là dữ liệu mà **có thể chứa lệnh**. Ta dùng ví dụ vô hại để thấy cơ chế; file lạ có thể chứa lệnh có hại.
- **Quy tắc đọc:** chỉ nạp file `.joblib` hoặc `.pkl` do **chính bạn hoặc nguồn bạn tin** tạo ra; đừng nạp file tải về từ người lạ. Nếu cần trao đổi với bên ngoài, dùng định dạng chỉ chứa dữ liệu (như ONNX hoặc `skops`) hoặc xuất ra hệ số dạng JSON.

**Mang về chặng 5:** `joblib.load` chạy code; chỉ nạp file mình tin.

## Chặng 5: Hàm nạp mô hình có kiểm tra

**Câu hỏi:** gói các kiểm tra trên thành một hàm duy nhất, dùng được cho mọi lần nạp?

In [7]:
import warnings
def load_model_safely(model_path, meta_path, golden_path):
    with open(meta_path, encoding="utf-8") as f:
        info = json.load(f)
    notes = []
    if info["sklearn_version"] != sklearn.__version__:
        notes.append(f"sklearn differs: trained on {info['sklearn_version']}, running {sklearn.__version__}")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        m = joblib.load(model_path)
    notes += [str(w.message)[:80] for w in caught]
    rows = pd.read_json(golden_path, orient="records")
    expected = rows.pop("expected_proba").to_numpy()
    diff = float(np.abs(m.predict_proba(rows[info["input_columns"]])[:, 1] - expected).max())
    if diff > 1e-6:
        raise RuntimeError(f"golden examples differ by {diff:.2e}; do not use this model")
    return m, info, notes
m2, info2, notes2 = load_model_safely(path, meta_path, golden_path)
print("loaded:", info2["name"], "| notes:", notes2 if notes2 else "none")
print("files in the model folder:", sorted(os.listdir(workdir)))

loaded: customer_return_v1 | notes: none
files in the model folder: ['classifier_only.joblib', 'customer_model.joblib', 'customer_model.meta.json', 'golden_examples.json']


**Giải thích từng dòng**

- `info["sklearn_version"] != sklearn.__version__`: so phiên bản đang chạy với phiên bản lúc huấn luyện; chỉ **ghi chú** chứ chưa dừng.
- `warnings.catch_warnings(record=True)`: bắt các cảnh báo xảy ra lúc nạp (ví dụ sklearn báo "file tạo bằng phiên bản khác").
- `raise RuntimeError(...)`: nếu ví dụ vàng lệch thì **dừng hẳn**, không cho dùng mô hình.
- `return m, info, notes`: trả về mô hình, ghi chú và danh sách điều cần lưu ý.

**Cách đọc.**
- **Con số:** nạp **customer_return_v1** thành công, ghi chú: **không có**. Thư mục mô hình có **4** file: ['classifier_only.joblib', 'customer_model.joblib', 'customer_model.meta.json', 'golden_examples.json'].
- **Bức tranh:** thư mục đóng gói gồm bốn thứ: mô hình, ghi chú, ví dụ vàng, và (ở H2) lược đồ đầu vào. Hàm trên là cổng duy nhất để mô hình đi vào sử dụng.
- **Lưu ý:** `classifier_only.joblib` trong thư mục là file của ví dụ "làm sai" ở chặng 3; trong dự án thật sẽ không có nó.

**Mang về:** lưu cả pipeline, ghi chú phiên bản, ví dụ vàng, chỉ nạp file mình tin, và nạp qua một hàm có kiểm tra.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 0 | Lưu cả pipeline bằng joblib; nạp lại cho kết quả giống hệt. |
| 1 | Ghi chú đi kèm: ngày, phiên bản thư viện, cột đầu vào, điểm, số dòng. |
| 2 | Ví dụ vàng: bài kiểm tra rẻ nhất sau mỗi lần nạp. |
| 3 | Không bao giờ tách mô hình khỏi chế biến dữ liệu; lỗi chạy được mà sai là lỗi xấu nhất. |
| 4 | joblib.load chạy code; chỉ nạp file mình tin. |
| 5 | Một hàm nạp có kiểm tra phiên bản và ví dụ vàng làm cổng duy nhất. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Đưa công thức đầy đủ, không đưa nồi canh nóng | đóng gói pipeline | `joblib.dump(model, path)` |
| Nhãn dán lên hộp | siêu dữ liệu mô hình | `meta.json` |
| Nếm món mẫu trước khi bán | ví dụ vàng | `golden_examples.json` |
| Công thức thiếu bước sơ chế | quên pipeline | chỉ lưu `LogisticRegression` |
| Hộp có thể chứa lệnh | rủi ro pickle | `__reduce__` |
| Cổng kiểm tra duy nhất | hàm nạp an toàn | `load_model_safely` |

**Thử thách tự làm**

1. Ở chặng 0, thay `LogisticRegression` bằng `HistGradientBoostingClassifier` và lưu lại. File to hay nhỏ hơn? Ví dụ vàng còn qua không?
2. Ở chặng 2, thêm vào ví dụ vàng một dòng có ô trống ở `monthly_spend` và một dòng có `plan = 'vip'`. Mô hình xử lý thế nào?
3. Ở chặng 5, thêm kiểm tra `info['input_columns']` có khớp với cột của bảng mới không, và báo lỗi nếu thiếu cột.
4. Lưu mô hình với `joblib.dump(model, path, compress=3)`. File nhỏ đi bao nhiêu và thời gian nạp thay đổi ra sao?

**Checklist năng lực** (tự tick khi làm được)

- [ ] Lưu và nạp cả pipeline bằng joblib.
- [ ] Viết ghi chú phiên bản đi kèm file mô hình.
- [ ] Tạo và chạy ví dụ vàng sau khi nạp.
- [ ] Giải thích vì sao không được tách mô hình khỏi chế biến dữ liệu.
- [ ] Biết rủi ro khi nạp file không rõ nguồn.

## Phần thưởng (Tip pro): Sang H2: hàm dự đoán an toàn


Mô hình đã nạp được rồi, nhưng dữ liệu đến từ nhân viên có thể thiếu cột, sai kiểu, hoặc vô lý (tuổi âm). H2 dạy viết một **hàm dự đoán** kiểm tra đầu vào, trả về xác suất kèm lý do, và ghi lại nhật ký để theo dõi (H3).